In [1]:
import os
import numpy as np
from astropy.io import fits
from scipy.ndimage import rotate
from skimage.filters import threshold_otsu
from skimage.measure import regionprops, label

In [3]:

# --- Configuration ---
# Replace these with your actual folder paths
input_dir = 'VELOCITY_VDISP_FLUX_MAPS'
output_dir = 'VELOCITY_VDISP_FLUX_ROTATED'

In [4]:

# Ensure the output directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [5]:

# Get a list of all FITS files in the input directory
fits_files = [f for f in os.listdir(input_dir) if f.endswith('.fits')]

In [6]:

for filename in fits_files[:10]:
    input_path = os.path.join(input_dir, filename)
    output_path = os.path.join(output_dir, filename)

    # 1. Open the FITS file
    with fits.open(input_path) as hdul:
        # Assuming the data is in the Primary HDU (index 0)
        data = hdul[0].data
        header = hdul[0].header

        # The array is (3, Y, X). Extract the flux map (index 0)
        flux_map = data[0, :, :]

        # 2. Create a mask to identify the galaxy vs the background
        # Otsu's method finds an optimal threshold value automatically
        try:
            thresh = threshold_otsu(flux_map)
            mask = flux_map > thresh
        except ValueError:
            # If the image is completely blank or uniform, skip or use a basic mask
            mask = flux_map > 0

        # 3. Find the major axis angle
        # Label the mask (in case there are multiple disjointed artifacts, we want the main one)
        labeled_mask = label(mask)
        props = regionprops(labeled_mask, intensity_image=flux_map)

        if not props:
            print(f"Warning: No clear object found in {filename}. Skipping rotation.")
            hdul.writeto(output_path, overwrite=True)
            continue

        # Assume the largest region by area is the galaxy
        main_galaxy = max(props, key=lambda r: r.area)

        # 'orientation' is the angle between the Y-axis (rows) and the major axis
        # Returns values from -pi/2 to pi/2
        angle_rad = main_galaxy.orientation
        angle_deg = np.degrees(angle_rad)

        # To make the major axis horizontal (aligned with X-axis),
        # we rotate by 90 degrees minus the current orientation.
        # (Note: depending on the exact camera parity, you might occasionally
        # need to swap the sign to `angle_deg - 90`. This configuration works standardly).
        rotation_angle = 90 - angle_deg

        # 4. Rotate the 3D array simultaneously
        # axes=(1,2) ensures we only rotate the spatial Y-X axes, leaving the 3 layers alone
        # reshape=False keeps the original (3, Y, X) dimensions
        # order=0 enforces nearest-neighbor interpolation
        # cval=0.0 fills empty clipped regions with zeros
        rotated_data = rotate(
            data,
            angle=rotation_angle,
            axes=(1, 2),
            reshape=False,
            order=0,
            cval=0.0
        )

        # 5. Save the modified data back to the new folder
        # We use the original header, ignoring WCS implications as requested
        hdu_new = fits.PrimaryHDU(rotated_data, header=header)
        hdu_new.writeto(output_path, overwrite=True)

    print(f"Processed: {filename} | Rotated by: {rotation_angle:.2f} degrees")

Processed: TNG50-96-59-0-127.cube_maps.fits | Rotated by: 154.24 degrees
Processed: TNG50-96-576846-0-127.cube_maps.fits | Rotated by: 17.11 degrees
Processed: TNG50-97-677968-0-127.cube_maps.fits | Rotated by: 178.95 degrees
Processed: TNG50-96-572097-0-127.cube_maps.fits | Rotated by: 117.82 degrees
Processed: TNG50-89-314723-0-127.cube_maps.fits | Rotated by: 20.88 degrees
Processed: TNG50-96-574540-0-127.cube_maps.fits | Rotated by: 135.00 degrees
Processed: TNG50-96-588269-0-127.cube_maps.fits | Rotated by: 157.51 degrees
Processed: TNG50-97-251669-0-127.cube_maps.fits | Rotated by: 41.93 degrees
Processed: TNG50-97-593464-0-127.cube_maps.fits | Rotated by: 64.56 degrees
Processed: TNG50-97-446530-0-127.cube_maps.fits | Rotated by: 151.96 degrees
